In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import math

In [2]:
data = [
    ("i am a student", "je suis un étudiant"),
    ("i love machine learning", "j'aime l'apprentissage automatique"),
    ("this is my book", "c'est mon livre"),
    ("how are you", "comment ça va"),
    ("i like coffee", "j'aime le café"),
]

In [3]:
def build_vocab(sentences):
    vocab = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
    idx = 4
    for sent in sentences:
        for word in sent.split():
            if word not in vocab:
                vocab[word] = idx
                idx += 1
    return vocab


In [4]:
eng_vocab = build_vocab([x[0] for x in data])
fre_vocab = build_vocab([x[1] for x in data])

eng_size = len(eng_vocab)
fre_size = len(fre_vocab)

In [5]:
class TranslationDataset(Dataset):
    def __init__(self, data, src_vocab, tgt_vocab, max_len=10):
        self.data = data
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len = max_len

    def encode(self, sentence, vocab):
        tokens = sentence.split()
        ids = [vocab["<SOS>"]] + \
              [vocab.get(w, vocab["<UNK>"]) for w in tokens] + \
              [vocab["<EOS>"]]
        ids = ids[:self.max_len]
        ids += [vocab["<PAD>"]] * (self.max_len - len(ids))
        return torch.tensor(ids)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src, tgt = self.data[idx]
        return self.encode(src, self.src_vocab), self.encode(tgt, self.tgt_vocab)

In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000)/d_model))
        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [7]:
class TransformerMT(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=64, nhead=4, num_layers=2):
        super().__init__()

        self.src_embed = nn.Embedding(src_vocab, d_model)
        self.tgt_embed = nn.Embedding(tgt_vocab, d_model)

        self.pos_enc = PositionalEncoding(d_model)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=128,
            dropout=0.1
        )

        self.fc = nn.Linear(d_model, tgt_vocab)

    def forward(self, src, tgt):
        src = self.pos_enc(self.src_embed(src))
        tgt = self.pos_enc(self.tgt_embed(tgt))

        src = src.permute(1,0,2)
        tgt = tgt.permute(1,0,2)

        out = self.transformer(src, tgt)
        out = out.permute(1,0,2)

        return self.fc(out)

In [8]:
dataset = TranslationDataset(data, eng_vocab, fre_vocab)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

model = TransformerMT(eng_size, fre_size)
criterion = nn.CrossEntropyLoss(ignore_index=fre_vocab["<PAD>"])
optimizer = optim.Adam(model.parameters(), lr=0.001)

d:\GenerativeAi_And_Llms\.venv\Lib\site-packages\torch\nn\modules\transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


In [9]:
epochs = 300

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for src, tgt in loader:
        optimizer.zero_grad()

        output = model(src, tgt[:, :-1])

        loss = criterion(
            output.reshape(-1, fre_size),
            tgt[:, 1:].reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

Epoch 0, Loss: 9.0962
Epoch 50, Loss: 0.1653
Epoch 100, Loss: 0.0614
Epoch 150, Loss: 0.0295
Epoch 200, Loss: 0.0192
Epoch 250, Loss: 0.0126
